In [ ]:
# Databricks notebook source
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

# Define a generic schema to accommodate data from both sources
schema = StructType() \
    .add("timestamp", StringType()) \
    .add("zone", StringType()) \
    .add("price", DoubleType()) \
    .add("source", StringType())

# Read from Kafka
# Key changes: 1. Remove kafka.ssl.ca.pem  2. Use kafkashaded. prefix
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "<YOUR_AIVEN_BOOTSTRAP_SERVERS>")
    .option("subscribe", "energy-prices")
    .option("startingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config",
        'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
        'username="avnadmin" password="<YOUR_AIVEN_PASSWORD>";')
    # .option("kafka.ssl.ca.pem", """...""")  # Comment out or delete this line
    .load()
)

parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# Write to Delta table
query = (parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/energy_realtime/checkpoints")
    .toTable("energy_prices_realtime"))